# REVE: дообучение на размеченных людях новой больницы против калибровки порога (Турция) и кривая «число размеченных — качество»

| Поле | Значение |
|------|----------|
| **Автор** | Захар Санников (zs) |
| **Дата** | 2026-10-27 |
| **Гипотеза** | Г1 (ЭЭГ) |
| **Вопрос** | При переносе в новую больницу сейчас подстраивается только порог по 20 размеченным людям. Даёт ли больше дообучение самой модели? Сколько размеченных записей клиники нужно? |
| **Модель** | REVE v4 (`zakharsann/h1-reve-v4`, обучена на NMT), ансамбль запусков: оценка человека — среднее вероятностей запусков |
| **Данные** | Турецкий набор из общего набора Г1 (`zakharsann/h1-prep-shared`), 121 человек (50 эпилепсия, 71 норма); NMT val / eval того же набора — только для N = 0 (порог) и ориентира «без переноса» |
| **Часть A (20 выборок)** | R = 20 первых выборок по 20 человек (10 + 10) тем же кодом и seed, что в калибровке на месте (`calib`, `default_rng(0)`, k = 20, тот же порядок людей); проверка — на остальных 101 |
| **Часть B (кривая)** | P повторов: 50 человек стратифицированно откладываются на проверку (21 эпилепсия + 29 норма), адаптационные N = 10, 20, 30, 50, 70 — вложенные подмножества остальных 71 (поровну классов, пока хватает эпилепсии; N = 70 — 29 + 41). N = 0 — порог с отложенных 15 % NMT train (NMT val) без изменений. Разбиения одни для всех вариантов |
| **Варианты** | (а) только порог по адаптационным (Юден / recall 80 %); (б) дообучение головы; (в) голова + последние 2 блока трансформера (из 22), малая скорость. В части B (в) — для N = 20 и 50, остальные N — если укладывается в бюджет |
| **Порог после дообучения** | по оценкам адаптационных людей вне выборки: 5-кратная перекрёстная проверка внутри них; затем модель обучается на всех адаптационных и применяется к проверке с этим порогом |
| **Метрики** | ROC AUC, сбалансированная accuracy, precision, recall, специфичность, accuracy — среднее и 2,5–97,5 % по выборкам / повторам; парные разности (б) − (а), (в) − (а) |
| **Среда** | Kaggle, GPU T4, бюджет ≤ 2 ч |

## 1. Загрузчик общего набора (копия `h1_eeg/src/shared_data.py`)

In [ ]:
"""Общий набор Г1 (NMT + турецкий) — выход блокнота Kaggle zakharsann/h1-prep-shared.

Подключение в kernel-metadata.json: "kernel_sources": ["zakharsann/h1-prep-shared"].
Блокноты моделей копируют этот файл в начало (одна ячейка) и пользуются функциями ниже.

Формат (подробно — в описании блокнота h1_2026-10-26_zs_prep-shared.ipynb):
- h1_shared_meta.csv — по строке на запись: dataset, part, split, id, y, ..., shard200/off200/n200 (и shard256/... если есть);
- h1_shared_info.json — фильтр, множитель масштаба NMT, сколько минут NMT train сохранено, порядок каналов;
- fs200/<имя>.npy — плоские массивы float16: записи (19, n) в C-порядке подряд, запись = arr[off : off + 19*n].
Сигнал в мкВ, 19 каналов в порядке EL, средний референт, 0,1-75 Гц + режекция 50 Гц. Окна НЕ нормированы.
"""
import glob, json, os
from fractions import Fraction
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, scipy.signal as ss

EL = ["Fp1", "Fp2", "F3", "F4", "C3", "C4", "P3", "P4", "O1", "O2", "F7", "F8", "T3", "T4", "T5", "T6", "Fz", "Cz", "Pz"]


def find_root(base="/kaggle/input"):
    """Папка общего набора среди подключённых источников."""
    hits = glob.glob(f"{base}/**/h1_shared_meta.csv", recursive=True)
    assert hits, "общий набор не подключён: добавьте zakharsann/h1-prep-shared в kernel_sources"
    return os.path.dirname(hits[0])


def load_meta(root):
    """Таблица записей и описание набора."""
    meta = pd.read_csv(f"{root}/h1_shared_meta.csv")
    info = json.load(open(f"{root}/h1_shared_info.json", encoding="utf-8"))
    return meta, info


_SHARDS = {}
def _shard(root, fs, name):
    k = (root, fs, name)
    if k not in _SHARDS:
        _SHARDS[k] = np.load(f"{root}/fs{fs}/{name}", mmap_mode="r")
    return _SHARDS[k]


def read_record(root, row, fs=200):
    """Одна запись: (19, n) float16, мкВ. row — строка meta (Series или dict).
    Через mmap — для единичных записей; много записей читать через read_all (на /kaggle/input mmap очень медленный)."""
    n, off = int(row[f"n{fs}"]), int(row[f"off{fs}"])
    return np.array(_shard(root, fs, row[f"shard{fs}"])[off:off + 19 * n]).reshape(19, n)


def read_all(root, meta, fs=200, threads=4):
    """Все записи meta в словарь id -> (19, n) float16: каждый файл-кусок читается целиком последовательно, записи режутся в памяти."""
    def load(item):
        name, g = item
        arr = np.load(f"{root}/fs{fs}/{name}")
        return [(i, arr[o:o + 19 * n].reshape(19, n).copy()) for i, o, n in zip(g["id"], g[f"off{fs}"], g[f"n{fs}"])]
    out = {}
    with ThreadPoolExecutor(threads) as ex:
        for part in ex.map(load, list(meta.groupby(f"shard{fs}"))):
            out.update(part)
    return {i: out[i] for i in meta["id"]}


def windows(X, L, S=None):
    """Окна длиной L с шагом S (по умолчанию S = L, подряд): (k, 19, L)."""
    S = S or L
    k = (X.shape[-1] - L) // S + 1 if X.shape[-1] >= L else 0
    if k <= 0:
        return np.zeros((0,) + X.shape[:-1] + (L,), X.dtype)
    return np.lib.stride_tricks.sliding_window_view(X, L, axis=-1)[..., ::S, :][..., :k, :].swapaxes(0, -2).copy()


def random_windows(X, L, K, rng):
    """K окон длиной L со случайным началом (любой отсчёт) — свежие окна на каждой эпохе: (K, 19, L)."""
    n = X.shape[-1] - L + 1
    if n <= 0:
        return np.zeros((0,) + X.shape[:-1] + (L,), X.dtype)
    st = np.sort(rng.integers(0, n, K))
    return np.stack([X[..., s:s + L] for s in st])


def resample(X, fs_from, fs_to):
    """Пересчёт частоты (полифазный, как scipy.signal.resample_poly) по последней оси, float32 на выходе."""
    if fs_from == fs_to:
        return X
    fr = Fraction(int(fs_to), int(fs_from))
    return ss.resample_poly(np.asarray(X, np.float32), fr.numerator, fr.denominator, axis=-1).astype(np.float32)


def highpass(X, f, fs=200, order=4):
    """Дофильтровка сверху общей нижней границы 0,1 Гц (например, 0,3 Гц как у CBraMod): по всей записи, не по окну."""
    sos = ss.butter(order, f, btype="highpass", fs=fs, output="sos")
    return ss.sosfiltfilt(sos, np.asarray(X, np.float64), axis=-1).astype(np.float32)

## 2. Модель REVE и веса v4

Устройство как в REVE v2: основа `brain-bzh/reve-base` (22 блока) + голова (среднее по времени с сохранением каналов → 19·512 → 256 → 1).
Для дообучения последних блоков вход блока 21 кэшируется один раз (основа до него заморожена), дальше считаются только блоки 21–22 и голова — так 20 выборок × 6 дообучений укладываются в бюджет.

In [ ]:
import os, glob, time, copy, json
SMOKE = False   # True — проверка конвейера на процессоре с временными весами (основа REVE + случайная голова), цифры не смотреть
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedKFold
from transformers import AutoModel
DEV = "cuda" if torch.cuda.is_available() else "cpu"
OUT = "/kaggle/working"
T_NB = time.time(); END = T_NB + 110 * 60
left = lambda: (END - time.time()) / 60
SFREQ, SEG = 200, 10; L = SFREQ * SEG; CAP_EVAL = 3 if SMOKE else 60
N_LAYERS, K_LAST = 22, 2
AMP = dict(device_type="cuda", dtype=torch.float16, enabled=DEV == "cuda")

NEW = {"T3": "T7", "T4": "T8", "T5": "P7", "T6": "P8"}
pos_bank = AutoModel.from_pretrained("brain-bzh/reve-positions", trust_remote_code=True)
try:
    POS = pos_bank([NEW.get(e, e) for e in EL])
except Exception as e:
    print("новые названия не подошли:", e); POS = pos_bank(EL)
POS = POS.to(DEV)
MIRROR_PAIRS = {"Fp1": "Fp2", "F3": "F4", "C3": "C4", "P3": "P4", "O1": "O2", "F7": "F8", "T3": "T4", "T5": "T6"}
MIRROR_PAIRS.update({v: k for k, v in list(MIRROR_PAIRS.items())})
MIRROR = [EL.index(MIRROR_PAIRS.get(e, e)) for e in EL]

def rv_input(W):
    """Окно → общее СКО 1 по всем каналам и отсчётам, обрезка ±15 (как в REVE v2)."""
    x = W.astype(np.float32)
    x = x - x.mean(-1, keepdims=True)
    x = x / (x.std(axis=(1, 2), keepdims=True) + 1e-6)
    return torch.from_numpy(np.clip(x, -15, 15))

class RVNet(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()
        self.backbone = AutoModel.from_pretrained("brain-bzh/reve-base", trust_remote_code=True)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(19 * 512, 256), nn.GELU(), nn.Dropout(dropout), nn.Linear(256, 1))
    def forward(self, x):
        o = self.backbone(x, POS.expand(x.size(0), -1, -1))
        if isinstance(o, (tuple, list)): o = o[0]
        if hasattr(o, "last_hidden_state"): o = o.last_hidden_state
        assert o.dim() == 4 and o.shape[1] == 19, tuple(o.shape)
        return self.head(o.float().mean(2).flatten(1)).squeeze(-1)

class Tail(nn.Module):
    """Последние K_LAST блоков трансформера + голова; вход — кэш входа блока N_LAYERS-K_LAST, (B, 19*h, 512).
    Блок REVE = пара (attn, ff) с остатками: x = attn(x) + x; x = ff(x) + x; после блоков нормировки нет (final_layer = Identity)."""
    def __init__(self, m):
        super().__init__()
        self.layers = copy.deepcopy(m.backbone.transformer.layers[N_LAYERS - K_LAST:])
        self.head = copy.deepcopy(m.head)
    def feats(self, h):
        for attn, ff in self.layers:
            h = attn(h) + h
            h = ff(h) + h
        B, T, E = h.shape
        return h.view(B, 19, T // 19, E).float().mean(2).flatten(1)
    def forward(self, h):
        return self.head(self.feats(h)).squeeze(-1)

In [ ]:
# Веса REVE v4 — выход блокнота zakharsann/h1-reve-v4 (kernel_sources); формат — по описанию того блокнота
WEIGHTS = sorted((int(os.path.basename(p)[len("reve_v4_seed"):-3]), p) for p in glob.glob("/kaggle/input/**/reve_v4_seed*.pt", recursive=True))
if SMOKE and not WEIGHTS:      # временные веса только для проверки конвейера
    torch.manual_seed(0); torch.save(RVNet().state_dict(), f"{OUT}/tmp_seed0.pt"); WEIGHTS = [(0, f"{OUT}/tmp_seed0.pt")]
assert WEIGHTS, "веса v4 не подключены"
print("веса:", [(s, os.path.relpath(p, "/kaggle")) for s, p in WEIGHTS])

_SHOWN = set()
def load_v4(path):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    sd = ck.get("state_dict", ck.get("model", ck)) if isinstance(ck, dict) else ck
    if isinstance(ck, dict) and "config" in ck and path not in _SHOWN:
        _SHOWN.add(path); print(os.path.basename(path), {k: ck["config"].get(k) for k in ["seed", "best_epoch", "val_auc_12win_median"]})
    m = RVNet().to(DEV)
    miss, unexp = m.load_state_dict(sd, strict=False)
    assert not miss and not unexp, (miss[:5], unexp[:5])
    return m.eval()

def prep_record(X):
    """Общий набор: 0,1–75 Гц + 50 Гц; как в REVE v2 — дофильтровка сверху до 0,3 Гц по всей записи."""
    return highpass(X, 0.3)

## 3. Турецкий набор: окна 10 с, до 60 на запись равномерно; порядок людей — как в калибровке на месте

In [ ]:
ORDER = [('1_epilepsy (1).mat', 1), ('1_epilepsy (10).mat', 1), ('1_epilepsy (11).mat', 1), ('1_epilepsy (12).mat', 1), ('1_epilepsy (13).mat', 1), ('1_epilepsy (14).mat', 1), ('1_epilepsy (15).mat', 1), ('1_epilepsy (16).mat', 1), ('1_epilepsy (17).mat', 1), ('1_epilepsy (18).mat', 1), ('1_epilepsy (19).mat', 1), ('1_epilepsy (2).mat', 1), ('1_epilepsy (20).mat', 1), ('1_epilepsy (21).mat', 1), ('1_epilepsy (22).mat', 1), ('1_epilepsy (23).mat', 1), ('1_epilepsy (24).mat', 1), ('1_epilepsy (25).mat', 1), ('1_epilepsy (26).mat', 1), ('1_epilepsy (27).mat', 1), ('1_epilepsy (28).mat', 1), ('1_epilepsy (29).mat', 1), ('1_epilepsy (3).mat', 1), ('1_epilepsy (30).mat', 1), ('1_epilepsy (31).mat', 1), ('1_epilepsy (32).mat', 1), ('1_epilepsy (33).mat', 1), ('1_epilepsy (34).mat', 1), ('1_epilepsy (35).mat', 1), ('1_epilepsy (36).mat', 1), ('1_epilepsy (37).mat', 1), ('1_epilepsy (38).mat', 1), ('1_epilepsy (39).mat', 1), ('1_epilepsy (4).mat', 1), ('1_epilepsy (40).mat', 1), ('1_epilepsy (41).mat', 1), ('1_epilepsy (42).mat', 1), ('1_epilepsy (43).mat', 1), ('1_epilepsy (44).mat', 1), ('1_epilepsy (45).mat', 1), ('1_epilepsy (46).mat', 1), ('1_epilepsy (47).mat', 1), ('1_epilepsy (48).mat', 1), ('1_epilepsy (49).mat', 1), ('1_epilepsy (5).mat', 1), ('1_epilepsy (50).mat', 1), ('1_epilepsy (6).mat', 1), ('1_epilepsy (7).mat', 1), ('1_epilepsy (8).mat', 1), ('1_epilepsy (9).mat', 1), ('2_Normal (1).mat', 0), ('2_Normal (10).mat', 0), ('2_Normal (11).mat', 0), ('2_Normal (12).mat', 0), ('2_Normal (13).mat', 0), ('2_Normal (14).mat', 0), ('2_Normal (15).mat', 0), ('2_Normal (16).mat', 0), ('2_Normal (17).mat', 0), ('2_Normal (18).mat', 0), ('2_Normal (19).mat', 0), ('2_Normal (2).mat', 0), ('2_Normal (20).mat', 0), ('2_Normal (21).mat', 0), ('2_Normal (22).mat', 0), ('2_Normal (23).mat', 0), ('2_Normal (24).mat', 0), ('2_Normal (25).mat', 0), ('2_Normal (26).mat', 0), ('2_Normal (27).mat', 0), ('2_Normal (28).mat', 0), ('2_Normal (29).mat', 0), ('2_Normal (3).mat', 0), ('2_Normal (30).mat', 0), ('2_Normal (31).mat', 0), ('2_Normal (32).mat', 0), ('2_Normal (33).mat', 0), ('2_Normal (34).mat', 0), ('2_Normal (35).mat', 0), ('2_Normal (36).mat', 0), ('2_Normal (37).mat', 0), ('2_Normal (38).mat', 0), ('2_Normal (39).mat', 0), ('2_Normal (4).mat', 0), ('2_Normal (40).mat', 0), ('2_Normal (41).mat', 0), ('2_Normal (42).mat', 0), ('2_Normal (43).mat', 0), ('2_Normal (44).mat', 0), ('2_Normal (45).mat', 0), ('2_Normal (46).mat', 0), ('2_Normal (47).mat', 0), ('2_Normal (48).mat', 0), ('2_Normal (49).mat', 0), ('2_Normal (5).mat', 0), ('2_Normal (50).mat', 0), ('2_Normal (51).mat', 0), ('2_Normal (52).mat', 0), ('2_Normal (53).mat', 0), ('2_Normal (54).mat', 0), ('2_Normal (55).mat', 0), ('2_Normal (56).mat', 0), ('2_Normal (57).mat', 0), ('2_Normal (58).mat', 0), ('2_Normal (59).mat', 0), ('2_Normal (6).mat', 0), ('2_Normal (60).mat', 0), ('2_Normal (61).mat', 0), ('2_Normal (62).mat', 0), ('2_Normal (63).mat', 0), ('2_Normal (64).mat', 0), ('2_Normal (65).mat', 0), ('2_Normal (66).mat', 0), ('2_Normal (67).mat', 0), ('2_Normal (68).mat', 0), ('2_Normal (69).mat', 0), ('2_Normal (7).mat', 0), ('2_Normal (70).mat', 0), ('2_Normal (71).mat', 0), ('2_Normal (8).mat', 0), ('2_Normal (9).mat', 0)]
ROOT = find_root(); meta, info = load_meta(ROOT)
mt = meta[meta.dataset == "turkey"].set_index("id")
assert len(mt) == 121 and set(mt.index) == {i for i, _ in ORDER}
mt = mt.loc[[i for i, _ in ORDER]].reset_index()
y_all = np.array([v for _, v in ORDER]); assert (mt.y.to_numpy() == y_all).all()
RAW = read_all(ROOT, mt)
def spread(n, k):
    return np.unique(np.linspace(0, n - 1, k).astype(int)) if n > k else np.arange(n)
WIN = []
for i in mt.id:
    X = prep_record(RAW[i])            # дофильтровка как при обучении v4 (см. раздел 2)
    W = windows(X, L); WIN.append(W[spread(len(W), CAP_EVAL)].astype(np.float16))
del RAW
n_w = np.array([len(w) for w in WIN]); PERSON = np.repeat(np.arange(len(WIN)), n_w); START = np.r_[0, np.cumsum(n_w)[:-1]]
ALLW = np.concatenate(WIN); del WIN
print(f"Турция: {len(y_all)} чел. (эпилепсия {y_all.sum()}), окон {len(ALLW)}, на запись: медиана {int(np.median(n_w))}, мин {n_w.min()} | {time.time()-T_NB:.0f} с")
# NMT val (отложенные 15 % NMT train) и NMT eval — только исходная модель: порог для N = 0 и ориентир «без переноса»
mn = meta[meta.split.isin(["val", "eval"])].reset_index(drop=True)
if SMOKE: mn = mn.groupby(["split", "y"]).head(5).reset_index(drop=True)
RAWN = read_all(ROOT, mn)
NMTW = [windows(prep_record(RAWN[i]), L) for i in mn.id]; del RAWN
NMTW = [w[spread(len(w), CAP_EVAL)].astype(np.float16) for w in NMTW]
print("NMT:", mn.groupby("split").y.agg(["size", "sum"]).to_dict(), f"| {time.time()-T_NB:.0f} с")

## 4. Разбиения: часть A — выборки `calib` (seed 0, k = 20, первые 20); часть B — повторы с общими отложенными 50

In [ ]:
R, K = 20, 20
r = np.random.default_rng(0); i1, i0 = np.where(y_all == 1)[0], np.where(y_all == 0)[0]
SPLITS = []      # dict(kind, rep, N, adapt, test)
for s_ in range(R):
    c = np.concatenate([r.choice(i1, K // 2, replace=False), r.choice(i0, K // 2, replace=False)])
    SPLITS.append(dict(kind="A", rep=s_, N=K, adapt=c, test=np.setdiff1d(np.arange(len(y_all)), c)))
print("часть A, первая выборка:", [ORDER[j][0] for j in SPLITS[0]["adapt"][:4]], "...")

P, N_TEST, NS = (2 if SMOKE else 10), 50, [10, 20, 30, 50, 70]
if SMOKE: SPLITS = SPLITS[:2]
for rep in range(P):
    g = np.random.default_rng(10_000 + rep)
    t1 = g.choice(i1, round(N_TEST * len(i1) / len(y_all)), replace=False); t0 = g.choice(i0, N_TEST - len(t1), replace=False)
    test = np.sort(np.r_[t1, t0])
    p1, p0 = g.permutation(np.setdiff1d(i1, t1)), g.permutation(np.setdiff1d(i0, t0))   # вложенные подмножества
    SPLITS.append(dict(kind="B", rep=rep, N=0, adapt=np.array([], int), test=test))
    for N in NS:
        n1 = min(N // 2, len(p1)); a_ = np.r_[p1[:n1], p0[:N - n1]]
        SPLITS.append(dict(kind="B", rep=rep, N=N, adapt=a_, test=test))
print("часть B: проверка", len(SPLITS[-1]["test"]), "(эпилепсия", int(y_all[SPLITS[-1]["test"]].sum()), ") | пул", len(y_all) - N_TEST,
      "| N:", {N: f"{int(y_all[d['adapt']].sum())}+{int((1 - y_all[d['adapt']]).sum())}" for d in SPLITS[-6:] for N in [d["N"]]})
pd.DataFrame([{"kind": d["kind"], "rep": d["rep"], "N": d["N"], "part": part, "id": ORDER[j][0], "y": int(y_all[j])}
              for d in SPLITS for part in ["adapt", "test"] for j in d[part] if not (d["kind"] == "B" and part == "test" and d["N"] > 0)]) \
  .to_csv(f"{OUT}/reve_fewshot_splits.csv", index=False)

## 5. Кэш входа последних блоков (по каждому запуску): чистые окна + 2 аугментированные копии

Аугментации — как при обучении REVE v2: шум 0,1·СКО, зеркальная замена полушарий (p = 0,5), зануление 1–2 каналов (p = 0,5), сдвиг до 2 с. Копии делаются один раз (одни и те же для всех запусков); на каждом шаге дообучения для окна случайно берётся одна из трёх версий.
Проверка: путь «кэш → последние блоки → голова» должен давать те же вероятности, что вся модель.

In [ ]:
N_AUG = 2
def augment(x, g):
    B = x.shape[0]
    x = x + 0.1 * torch.randn(x.shape, generator=g)
    m = torch.rand(B, generator=g) < 0.5
    x[m] = x[m][:, MIRROR]
    for b in range(B):
        if torch.rand(1, generator=g).item() < 0.5:
            k = int(torch.randint(1, 3, (1,), generator=g)); x[b, torch.randperm(19, generator=g)[:k]] = 0
        x[b] = torch.roll(x[b], int(torch.randint(-2 * SFREQ, 2 * SFREQ + 1, (1,), generator=g)), dims=-1)
    return x

def build_cache(model, bs=64):
    """(N_AUG+1, окна, 19*h, 512) float16 на CPU; копия 0 — чистые окна."""
    cap = {}
    hk = model.backbone.transformer.layers[N_LAYERS - K_LAST][0].register_forward_pre_hook(lambda mod, a: cap.__setitem__("h", a[0].detach()))
    g = torch.Generator().manual_seed(123); H = None
    model.eval()
    with torch.no_grad(), torch.autocast(**AMP):
        for v in range(N_AUG + 1):
            for b in range(0, len(ALLW), bs):
                x = rv_input(ALLW[b:b + bs])
                if v: x = augment(x, g)
                model(x.to(DEV)); h = cap["h"].half().cpu()
                if H is None: H = torch.empty((N_AUG + 1, len(ALLW)) + tuple(h.shape[1:]), dtype=torch.float16)
                H[v, b:b + len(h)] = h
    hk.remove()
    return H

def tail_probs(tail, H0, idx, bs=256):
    """Вероятности окон idx по чистому кэшу."""
    tail.eval(); out = []
    with torch.no_grad(), torch.autocast(**AMP):
        for b in range(0, len(idx), bs):
            out.append(torch.sigmoid(tail(H0[idx[b:b + bs]].to(DEV, non_blocking=True)).float()).cpu().numpy())
    return np.concatenate(out)

def person_scores(pw, people):
    """Медиана вероятностей окон человека (как в REVE v2)."""
    return np.array([np.median(pw[START[p]:START[p] + n_w[p]]) for p in people])

def win_idx(people):
    return np.concatenate([np.arange(START[p], START[p] + n_w[p]) for p in people])

## 6. Дообучение на адаптационных людях

Настройки заданы заранее (подбирать их не на чем — 20 людей уходят на обучение и порог):
- **(б) голова:** AdamW, скорость 1·10⁻⁴, weight decay 0,05, 15 эпох, в эпохе по 30 случайных окон каждого человека (случайная версия: чистая или одна из 2 аугментированных), пакет 32; основа заморожена (голова считает по выходу основы, который кэшируется один раз).
- **(в) голова + блоки 21–22:** голова 1·10⁻⁴, блоки 1·10⁻⁵, 8 эпох, остальное так же.
- Старт — веса v4 соответствующего запуска; 5-кратная проверка внутри адаптационных — стратифицированно.

In [ ]:
CFG = {"б": dict(lr_head=1e-4, lr_blocks=None, epochs=15), "в": dict(lr_head=1e-4, lr_blocks=1e-5, epochs=8)}
if SMOKE:
    for c_ in CFG.values(): c_["epochs"] = 1
WPP, BS = 30, 32

def finetune(base_tail, H, Fh, people, variant, seed):
    """people — индексы обучающих людей. (б): обучается только голова по кэшу признаков Fh; (в): блоки + голова по кэшу H."""
    c = CFG[variant]; g = np.random.default_rng(seed); torch.manual_seed(seed)
    tail = copy.deepcopy(base_tail).to(DEV)
    groups = [{"params": tail.head.parameters(), "lr": c["lr_head"]}]
    if c["lr_blocks"]:
        groups.append({"params": tail.layers.parameters(), "lr": c["lr_blocks"]})
    else:
        for p in tail.layers.parameters(): p.requires_grad_(False)
    opt = torch.optim.AdamW(groups, weight_decay=0.05)
    scaler = torch.amp.GradScaler("cuda", enabled=DEV == "cuda")
    for ep in range(c["epochs"]):
        wi = np.concatenate([START[p] + g.choice(n_w[p], min(WPP, n_w[p]), replace=False) for p in people]); g.shuffle(wi)
        ver = g.integers(0, N_AUG + 1, len(wi)); yw = y_all[PERSON[wi]]
        wi_t, ver_t = torch.from_numpy(wi), torch.from_numpy(ver)
        tail.train()
        if not c["lr_blocks"]: tail.layers.eval()
        for b in range(0, len(wi), BS):
            j, v = wi_t[b:b + BS], ver_t[b:b + BS]
            t = torch.tensor(yw[b:b + BS], device=DEV, dtype=torch.float32)
            with torch.autocast(**AMP):
                if c["lr_blocks"]:
                    out = tail(H[v, j].to(DEV, non_blocking=True))
                else:
                    out = tail.head(Fh[v, j].to(DEV, non_blocking=True).float()).squeeze(-1)
            loss = F.binary_cross_entropy_with_logits(out.float(), t)
            opt.zero_grad()
            if torch.isfinite(loss):
                scaler.scale(loss).backward(); scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_([p for p in tail.parameters() if p.requires_grad], 1.0)
                scaler.step(opt); scaler.update()
    return tail

def head_probs(tail, Fh0, idx, bs=1024):
    tail.eval(); out = []
    with torch.no_grad():
        for b in range(0, len(idx), bs):
            out.append(torch.sigmoid(tail.head(Fh0[idx[b:b + bs]].to(DEV).float()).squeeze(-1)).cpu().numpy())
    return np.concatenate(out)

def score(tail, H, Fh, people, variant):
    idx = win_idx(people)
    pw = np.zeros(len(ALLW), np.float32)
    pw[idx] = head_probs(tail, Fh[0], idx) if variant == "б" else tail_probs(tail, H[0], idx)
    return person_scores(pw, people)

## 7. Прогон по каждому запуску v4: кэш, исходные оценки (а) и NMT, затем (б)/(в) — 5-кратная проверка внутри адаптационных + обучение на всех

Порядок работ: часть A (б, в), часть B (б) на всех N, (в) на N = 20 и 50; (в) на N = 10, 30, 70 — только если по замеру на запуске 0 укладывается во все запуски.

In [ ]:
SC = {}      # (seed, variant, номер разбиения) -> dict(oof=оценки адаптационных (для (а) — исходные), test=оценки проверки)
TIMES, NMT_P = [], {}
OPT_V = None  # (в) на N = 10, 30, 70 части B: решается после запуска 0

def run_split(seed, si, variant, base_ps, H, Fh):
    d = SPLITS[si]; c, rest = d["adapt"], d["test"]
    if variant == "а" or d["N"] == 0:
        SC[(seed, variant, si)] = dict(oof=base_ps[c], test=base_ps[rest]); return
    ts = time.time(); oof = np.zeros(len(c))
    for f, (a, b) in enumerate(StratifiedKFold(5, shuffle=True, random_state=si).split(c, y_all[c])):
        t = finetune(base_tail, H, Fh, c[a], variant, seed=100_000 * seed + 10 * si + f)
        oof[b] = score(t, H, Fh, c[b], variant); del t
    t = finetune(base_tail, H, Fh, c, variant, seed=100_000 * seed + 10 * si + 9)
    SC[(seed, variant, si)] = dict(oof=oof, test=score(t, H, Fh, rest, variant)); del t
    TIMES.append(dict(seed=seed, variant=variant, kind=d["kind"], N=d["N"], split=si, sec=time.time() - ts))

def nmt_probs(model, bs=64):
    model.eval(); out = []
    with torch.no_grad(), torch.autocast(**AMP):
        for W in NMTW:
            out.append(np.median(np.concatenate([torch.sigmoid(model(rv_input(W[b:b + bs]).to(DEV)).float()).cpu().numpy() for b in range(0, len(W), bs)])))
    return np.array(out)

for seed, path in WEIGHTS:
    t0 = time.time()
    model = load_v4(path)
    NMT_P[seed] = nmt_probs(model); t_nmt = time.time() - t0
    H = build_cache(model); t_cache = time.time() - t0 - t_nmt
    base_tail = Tail(model).to(DEV)
    with torch.no_grad(), torch.autocast(**AMP):
        p_full = torch.sigmoid(model(rv_input(ALLW[:64]).to(DEV)).float()).cpu().numpy()
    del model; torch.cuda.empty_cache()
    p_tail = tail_probs(base_tail, H[0], np.arange(64))
    print(f"запуск {seed}: NMT {t_nmt:.0f} с, кэш {tuple(H.shape)} {t_cache:.0f} с; расхождение полной модели и пути через кэш: {np.abs(p_full - p_tail).max():.4f}", flush=True)
    assert np.abs(p_full - p_tail).max() < 0.02
    Fh = torch.empty((N_AUG + 1, len(ALLW), 19 * 512), dtype=torch.float16)
    base_tail.eval()
    with torch.no_grad(), torch.autocast(**AMP):
        for v in range(N_AUG + 1):
            for b in range(0, len(ALLW), 256):
                Fh[v, b:b + 256] = base_tail.feats(H[v, b:b + 256].to(DEV)).half().cpu()
    base_ps = person_scores(tail_probs(base_tail, H[0], np.arange(len(ALLW))), np.arange(len(y_all)))
    ye = mn.y.to_numpy()[mn.split.to_numpy() == "eval"]
    print(f"запуск {seed}: исходная модель — Турция 121: {roc_auc_score(y_all, base_ps):.3f}, NMT eval: {roc_auc_score(ye, NMT_P[seed][mn.split.to_numpy() == 'eval']):.3f} | осталось {left():.0f} мин", flush=True)
    for si in range(len(SPLITS)):
        run_split(seed, si, "а", base_ps, H, Fh)
    main = [si for si, d in enumerate(SPLITS) if d["kind"] == "A" or d["N"] > 0]
    for si in main:
        run_split(seed, si, "б", base_ps, H, Fh)
    tv = pd.DataFrame(TIMES); print(f"  запуск {seed}: (б) готово, {tv[tv.seed == seed].sec.sum() / 60:.1f} мин | осталось {left():.0f} мин", flush=True)
    for si in main:
        if SPLITS[si]["kind"] == "A" or SPLITS[si]["N"] in (20, 50):
            run_split(seed, si, "в", base_ps, H, Fh)
    tv = pd.DataFrame(TIMES); tv = tv[(tv.seed == seed) & (tv.variant == "в")]
    if OPT_V is None:
        per = tv.sec.sum() / len(tv) * 1.0; n_opt = sum(SPLITS[si]["N"] in (10, 30, 70) for si in main)
        sec_per_N = tv.groupby("N").sec.mean().to_dict()                     # время растёт с N: оценка по 20 и 50 с линейной поправкой
        est = lambda N: sec_per_N[20] + (sec_per_N[50] - sec_per_N[20]) * (N - 20) / 30
        opt_min = sum(est(SPLITS[si]["N"]) for si in main if SPLITS[si]["N"] in (10, 30, 70)) / 60
        seed_min = (time.time() - t0) / 60
        need = opt_min * len(WEIGHTS) + seed_min * (len(WEIGHTS) - 1) + 3
        OPT_V = need < left()
        print(f"  (в) на N = 10, 30, 70: ≈{opt_min:.1f} мин на запуск; запуск без них {seed_min:.1f} мин; нужно ≈{need:.0f} из {left():.0f} мин -> {'да' if OPT_V else 'нет'}", flush=True)
    if OPT_V:
        for si in main:
            if SPLITS[si]["N"] in (10, 30, 70):
                run_split(seed, si, "в", base_ps, H, Fh)
    print(f"запуск {seed}: всего {(time.time() - t0) / 60:.1f} мин | осталось {left():.0f} мин", flush=True)
    del H, Fh, base_tail; torch.cuda.empty_cache()
SEEDS_DONE = sorted({k[0] for k in SC})
print("запуски:", SEEDS_DONE, f"| весь прогон {(time.time()-T_NB)/60:.1f} мин")

Вывод разделов 2–7 (Kaggle, GPU T4, 3 запуска v4):

```
0.00s - to python to disable frozen modules.
0.00s - to python to disable frozen modules.
веса: [(0, 'input/notebooks/zakharsann/h1-reve-v4/reve_v4_seed0.pt'), (1, 'input/notebooks/zakharsann/h1-reve-v4/reve_v4_seed1.pt'), (2, 'input/notebooks/zakharsann/h1-reve-v4/reve_v4_seed2.pt')]
Турция: 121 чел. (эпилепсия 50), окон 7240, на запись: медиана 60, мин 51 | 38 с
NMT: {'size': {'eval': 185, 'val': 335}, 'sum': {'eval': 90, 'val': 49}} | 116 с
часть A, первая выборка: ['1_epilepsy (40).mat', '1_epilepsy (46).mat', '1_epilepsy (33).mat', '1_epilepsy (29).mat'] ...
часть B: проверка 50 (эпилепсия 21 ) | пул 71 | N: {0: '0+0', 10: '5+5', 20: '10+10', 30: '15+15', 50: '25+25', 70: '29+41'}
reve_v4_seed0.pt {'seed': 0, 'best_epoch': 3, 'val_auc_12win_median': 0.8787}
запуск 0: NMT 122 с, кэш (3, 7240, 209, 512) 91 с; расхождение полной модели и пути через кэш: 0.0002
запуск 0: исходная модель — Турция 121: 0.899, NMT eval: 0.863 | осталось 104 мин
  запуск 0: (б) готово, 9.7 мин | осталось 95 мин
  (в) на N = 10, 30, 70: ≈28.7 мин на запуск; запуск без них 42.5 мин; нужно ≈174 из 66 мин -> нет
запуск 0: всего 42.5 мин | осталось 66 мин
reve_v4_seed1.pt {'seed': 1, 'best_epoch': 1, 'val_auc_12win_median': 0.8869}
запуск 1: NMT 116 с, кэш (3, 7240, 209, 512) 91 с; расхождение полной модели и пути через кэш: 0.0001
запуск 1: исходная модель — Турция 121: 0.866, NMT eval: 0.891 | осталось 62 мин
  запуск 1: (б) готово, 9.7 мин | осталось 52 мин
запуск 1: всего 42.5 мин | осталось 23 мин
reve_v4_seed2.pt {'seed': 2, 'best_epoch': 2, 'val_auc_12win_median': 0.8888}
запуск 2: NMT 118 с, кэш (3, 7240, 209, 512) 91 с; расхождение полной модели и пути через кэш: 0.0002
запуск 2: исходная модель — Турция 121: 0.890, NMT eval: 0.875 | осталось 19 мин
  запуск 2: (б) готово, 9.7 мин | осталось 10 мин
запуск 2: всего 42.6 мин | осталось -20 мин
запуски: [0, 1, 2] | весь прогон 129.6 мин
```

## 8. Метрики: часть A (101 человек, 20 выборок) и часть B (общие 50, кривая по N), парные разности, csv, график

In [ ]:
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
def thresholds(y, s):
    fpr, tpr, th = roc_curve(y, s)
    return {"Юден": th[np.argmax(tpr - fpr)], "recall 80 %": th[np.argmax(tpr >= 0.8)]}
def metr(y, s, t):
    pr = s >= t; tp = (pr & (y == 1)).sum(); tn = (~pr & (y == 0)).sum(); fp = (pr & (y == 0)).sum(); fn = (~pr & (y == 1)).sum()
    rec, spec = tp / (tp + fn), tn / (tn + fp)
    return dict(bacc=(rec + spec) / 2, prec=tp / (tp + fp) if tp + fp else np.nan, rec=rec, spec=spec, acc=(tp + tn) / len(y))

# порог N = 0: отложенные 15 % NMT train, ансамбль запусков (среднее вероятностей); NMT eval — ориентир «без переноса»
is_val, is_ev = mn.split.to_numpy() == "val", mn.split.to_numpy() == "eval"
p_nmt = np.mean([NMT_P[sd] for sd in SEEDS_DONE], 0); yn = mn.y.to_numpy()
THR_NMT = thresholds(yn[is_val], p_nmt[is_val])
nmt_ref = dict(auc=roc_auc_score(yn[is_ev], p_nmt[is_ev]), **{f"{k}|{tn}": v for tn, t in THR_NMT.items() for k, v in metr(yn[is_ev], p_nmt[is_ev], t).items()})
print("NMT eval (исходная модель, порог с NMT val):", {k: round(v, 3) for k, v in nmt_ref.items()})

ens = lambda v, si, k: np.mean([SC[(sd, v, si)][k] for sd in SEEDS_DONE], 0)   # ансамбль: среднее вероятностей запусков
NAMES = {"а": "(а) только порог", "б": "(б) дообучение головы", "в": "(в) голова + 2 последних блока"}
rows, pp = [], []
for si, d in enumerate(SPLITS):
    c, rest = d["adapt"], d["test"]
    for v in "абв":
        if (SEEDS_DONE[0], v, si) not in SC and d["N"] > 0: continue
        vv = "а" if d["N"] == 0 else v
        o, te = ens(vv, si, "oof"), ens(vv, si, "test")
        thr = THR_NMT if d["N"] == 0 else thresholds(y_all[c], o)
        for tn, t in thr.items():
            rows.append(dict(kind=d["kind"], rep=d["rep"], N=d["N"], split=si, variant=v, threshold=tn, thr=t, auc=roc_auc_score(y_all[rest], te),
                             auc_adapt_oof=roc_auc_score(y_all[c], o) if d["N"] else np.nan, **metr(y_all[rest], te, t)))
        if d["N"] == 0: continue
        for sd in SEEDS_DONE + ["ens"]:
            oo = o if sd == "ens" else SC[(sd, v, si)]["oof"]; tt = te if sd == "ens" else SC[(sd, v, si)]["test"]
            pp.append(pd.DataFrame({"kind": d["kind"], "rep": d["rep"], "N": d["N"], "variant": v, "seed": str(sd), "id": [ORDER[j][0] for j in np.r_[c, rest]],
                                    "y": y_all[np.r_[c, rest]], "part": ["adapt"] * len(c) + ["test"] * len(rest), "score": np.r_[oo, tt]}))
D = pd.DataFrame(rows)
D.to_csv(f"{OUT}/reve_fewshot_per_split.csv", index=False)
pd.concat(pp).to_csv(f"{OUT}/reve_fewshot_scores.csv.gz", index=False)
pd.DataFrame(TIMES).to_csv(f"{OUT}/reve_fewshot_times.csv", index=False)

COLS = {"auc": "ROC AUC", "bacc": "сбаланс. accuracy", "prec": "precision", "rec": "recall", "spec": "специфичность", "acc": "accuracy"}
f = lambda x: f"{np.nanmean(x):.3f} ({np.nanpercentile(x, 2.5):.2f}–{np.nanpercentile(x, 97.5):.2f})"
A = D[D.kind == "A"]
summ = pd.DataFrame([{"порог": tn, "вариант": NAMES[v], **{lab: f(g[c]) for c, lab in COLS.items()}}
                     for tn in ["Юден", "recall 80 %"] for v in "абв" for g in [A[(A.threshold == tn) & (A.variant == v)]]])
summ.to_csv(f"{OUT}/reve_fewshot_summary.csv", index=False)
print(f"\nЧасть A. Ансамбль запусков {SEEDS_DONE}; среднее по {R} выборкам (2,5–97,5 % выборок), проверка на 101:")
print(summ.to_string(index=False))

def paired(G, keys):
    out = []
    for key, g in G.groupby(keys):
        a = g[g.variant == "а"].set_index("split")
        for v in "бв":
            b = g[g.variant == v].set_index("split")
            if not len(b): continue
            row = dict(zip(keys, key if isinstance(key, tuple) else (key,))); row["разность"] = f"{NAMES[v][:3]} − (а)"; row["n"] = len(b)
            for c, lab in COLS.items():
                x = (b[c] - a.loc[b.index, c]).to_numpy()
                row[lab] = f"{np.nanmean(x):+.3f} ({np.nanpercentile(x, 2.5):+.2f}…{np.nanpercentile(x, 97.5):+.2f}); >0 в {int((x > 0).sum())}/{len(x)}"
            out.append(row)
    return pd.DataFrame(out)
diffA = paired(A, ["threshold"]); diffA.insert(0, "часть", "A (101, 20 выборок)")
B = D[D.kind == "B"]
diffB = paired(B[B.N > 0], ["threshold", "N"]); diffB.insert(0, "часть", "B (общие 50)")
pd.concat([diffA, diffB]).to_csv(f"{OUT}/reve_fewshot_paired_differences.csv", index=False)
print("\nПарные разности, часть A:"); print(diffA.to_string(index=False))

curve = (B.groupby(["N", "variant", "threshold"])[list(COLS)]
          .agg([("mean", "mean"), ("lo", lambda x: np.nanpercentile(x, 2.5)), ("hi", lambda x: np.nanpercentile(x, 97.5)), ("n", "count")]))
curve.columns = [f"{a}_{b}" for a, b in curve.columns]; curve = curve.reset_index()
curve.to_csv(f"{OUT}/reve_fewshot_curve.csv", index=False)
show = curve[[c for c in curve.columns if c.endswith("_mean") or c in ("N", "variant", "threshold", "auc_n")]].round(3)
print(f"\nЧасть B (кривая), среднее по {P} повторам, проверка на общих 50:"); print(show.to_string(index=False))
print("\nПарные разности, часть B:"); print(diffB.to_string(index=False))

# --- график: N — ось X; ROC AUC, сбаланс. accuracy (Юден), recall и специфичность (порог recall 80 %) ---
COL = {"а": "#2a78d6", "б": "#eb6834", "в": "#1baf7a"}; MK = {"а": "o", "б": "s", "в": "^"}
panels = [("auc", "Юден", "ROC AUC"), ("bacc", "Юден", "Сбаланс. accuracy (порог Юдена)"),
          ("rec", "recall 80 %", "Recall (порог «recall ≥ 80 %»)"), ("spec", "recall 80 %", "Специфичность (порог «recall ≥ 80 %»)")]
fig, axs = plt.subplots(2, 2, figsize=(11, 8), sharex=True)
for ax, (m, tn, title) in zip(axs.flat, panels):
    for v in "абв":
        q = curve[(curve.variant == v) & (curve.threshold == tn)].sort_values("N")
        if v != "а": q = q[q[f"{m}_n"] > 0]
        ax.fill_between(q.N, q[f"{m}_lo"], q[f"{m}_hi"], color=COL[v], alpha=0.12, lw=0)
        ax.plot(q.N, q[f"{m}_mean"], color=COL[v], lw=2, marker=MK[v], ms=8, mec="white", mew=1.5, label=NAMES[v])
    ref = nmt_ref["auc"] if m == "auc" else nmt_ref[f"{m}|{tn}"]
    ax.axhline(ref, color="#52514e", lw=1, ls="--"); ax.text(70, ref, " NMT eval", va="bottom", ha="right", fontsize=9, color="#52514e")
    ax.set_title(title, fontsize=11, loc="left"); ax.grid(axis="y", color="#e5e4e0", lw=0.8); ax.set_axisbelow(True)
    for sp in ("top", "right"): ax.spines[sp].set_visible(False)
for ax in axs[1]: ax.set_xlabel("N — размеченных людей новой больницы для адаптации"); ax.set_xticks([0] + NS)
h_, l_ = axs[0, 0].get_legend_handles_labels(); fig.legend(h_, l_, loc="lower center", ncol=3, frameon=False, fontsize=10)
fig.suptitle(f"REVE v4 → турецкий набор: качество на общих 50 проверочных в зависимости от N\n"
             f"среднее и 2,5–97,5 % по {P} повторам; N = 0 — порог перенесён с NMT; пунктир — NMT eval без переноса", fontsize=11, x=0.01, ha="left")
fig.tight_layout(rect=(0, 0.05, 1, 1)); fig.savefig(f"{OUT}/reve_fewshot_curve.png", dpi=150); plt.show()
print(f"всего {(time.time()-T_NB)/60:.1f} мин")

Вывод раздела 8:

```
NMT eval (исходная модель, порог с NMT val): {'auc': np.float64(0.878), 'bacc|Юден': np.float64(0.79), 'prec|Юден': np.float64(0.763), 'rec|Юден': np.float64(0.822), 'spec|Юден': np.float64(0.758), 'acc|Юден': np.float64(0.789), 'bacc|recall 80 %': np.float64(0.761), 'prec|recall 80 %': np.float64(0.78), 'rec|recall 80 %': np.float64(0.711), 'spec|recall 80 %': np.float64(0.811), 'acc|recall 80 %': np.float64(0.762)}
Часть A. Ансамбль запусков [0, 1, 2]; среднее по 20 выборкам (2,5–97,5 % выборок), проверка на 101:
      порог                        вариант           ROC AUC сбаланс. accuracy         precision            recall     специфичность          accuracy
       Юден               (а) только порог 0.889 (0.87–0.92) 0.802 (0.71–0.84) 0.879 (0.74–1.00) 0.682 (0.42–0.84) 0.922 (0.80–1.00) 0.827 (0.77–0.87)
       Юден          (б) дообучение головы 0.926 (0.85–0.97) 0.817 (0.69–0.91) 0.852 (0.66–0.98) 0.737 (0.39–0.96) 0.896 (0.70–0.99) 0.833 (0.75–0.92)
       Юден (в) голова + 2 последних блока 0.932 (0.87–0.98) 0.823 (0.72–0.90) 0.838 (0.64–0.98) 0.769 (0.45–0.96) 0.878 (0.66–0.99) 0.835 (0.75–0.90)
recall 80 %               (а) только порог 0.889 (0.87–0.92) 0.807 (0.75–0.84) 0.815 (0.61–0.98) 0.751 (0.54–0.91) 0.862 (0.61–0.99) 0.818 (0.73–0.85)
recall 80 %          (б) дообучение головы 0.926 (0.85–0.97) 0.829 (0.72–0.91) 0.819 (0.64–0.98) 0.796 (0.49–0.97) 0.861 (0.65–0.99) 0.836 (0.75–0.92)
recall 80 % (в) голова + 2 последних блока 0.932 (0.87–0.98) 0.824 (0.72–0.90) 0.827 (0.62–1.00) 0.785 (0.43–0.99) 0.863 (0.61–1.00) 0.832 (0.74–0.90)
Парные разности, часть A:
              часть   threshold  разность  n                          ROC AUC                сбаланс. accuracy                        precision                           recall                    специфичность                         accuracy
A (101, 20 выборок) recall 80 % (б) − (а) 20 +0.037 (-0.05…+0.09); >0 в 18/20 +0.022 (-0.12…+0.12); >0 в 15/20 +0.004 (-0.23…+0.20); >0 в 11/20 +0.045 (-0.33…+0.35); >0 в 13/20  -0.001 (-0.25…+0.22); >0 в 9/20 +0.017 (-0.08…+0.12); >0 в 12/20
A (101, 20 выборок) recall 80 % (в) − (а) 20 +0.043 (-0.04…+0.09); >0 в 19/20 +0.017 (-0.12…+0.11); >0 в 12/20 +0.011 (-0.21…+0.23); >0 в 10/20 +0.034 (-0.36…+0.34); >0 в 12/20 +0.001 (-0.27…+0.23); >0 в 11/20 +0.014 (-0.09…+0.12); >0 в 13/20
A (101, 20 выборок)        Юден (б) − (а) 20 +0.037 (-0.05…+0.09); >0 в 18/20 +0.014 (-0.14…+0.12); >0 в 13/20  -0.027 (-0.28…+0.20); >0 в 9/20 +0.055 (-0.45…+0.35); >0 в 14/20  -0.026 (-0.28…+0.17); >0 в 9/20  +0.006 (-0.09…+0.11); >0 в 9/20
A (101, 20 выборок)        Юден (в) − (а) 20 +0.043 (-0.04…+0.09); >0 в 19/20 +0.021 (-0.10…+0.11); >0 в 14/20 -0.042 (-0.29…+0.24); >0 в 10/20 +0.086 (-0.38…+0.42); >0 в 15/20  -0.044 (-0.30…+0.18); >0 в 8/20 +0.007 (-0.08…+0.10); >0 в 11/20
Часть B (кривая), среднее по 10 повторам, проверка на общих 50:
 N variant   threshold  auc_mean  auc_n  bacc_mean  prec_mean  rec_mean  spec_mean  acc_mean
 0       а recall 80 %     0.885     10      0.514      1.000     0.029      1.000     0.592
 0       а        Юден     0.885     10      0.531      1.000     0.062      1.000     0.606
 0       б recall 80 %     0.885     10      0.514      1.000     0.029      1.000     0.592
 0       б        Юден     0.885     10      0.531      1.000     0.062      1.000     0.606
 0       в recall 80 %     0.885     10      0.514      1.000     0.029      1.000     0.592
 0       в        Юден     0.885     10      0.531      1.000     0.062      1.000     0.606
10       а recall 80 %     0.885     10      0.802      0.756     0.838      0.766     0.796
10       а        Юден     0.885     10      0.836      0.854     0.786      0.886     0.844
10       б recall 80 %     0.870     10      0.753      0.715     0.824      0.683     0.742
10       б        Юден     0.870     10      0.754      0.800     0.681      0.828     0.766
20       а recall 80 %     0.885     10      0.789      0.722     0.848      0.731     0.780
20       а        Юден     0.885     10      0.824      0.804     0.810      0.838     0.826
20       б recall 80 %     0.893     10      0.804      0.808     0.776      0.831     0.808
20       б        Юден     0.893     10      0.806      0.821     0.743      0.869     0.816
20       в recall 80 %     0.905     10      0.801      0.792     0.771      0.831     0.806
20       в        Юден     0.905     10      0.801      0.792     0.771      0.831     0.806
30       а recall 80 %     0.885     10      0.823      0.817     0.805      0.841     0.826
30       а        Юден     0.885     10      0.828      0.800     0.819      0.838     0.830
30       б recall 80 %     0.907     10      0.774      0.731     0.786      0.762     0.772
30       б        Юден     0.907     10      0.782      0.734     0.795      0.769     0.780
50       а recall 80 %     0.885     10      0.818      0.764     0.833      0.803     0.816
50       а        Юден     0.885     10      0.812      0.810     0.776      0.848     0.818
50       б recall 80 %     0.929     10      0.846      0.826     0.819      0.872     0.850
50       б        Юден     0.929     10      0.834      0.805     0.824      0.845     0.836
50       в recall 80 %     0.935     10      0.823      0.854     0.752      0.893     0.834
50       в        Юден     0.935     10      0.834      0.848     0.781      0.886     0.842
70       а recall 80 %     0.885     10      0.818      0.767     0.829      0.807     0.816
70       а        Юден     0.885     10      0.816      0.798     0.800      0.831     0.818
70       б recall 80 %     0.930     10      0.820      0.813     0.786      0.855     0.826
70       б        Юден     0.930     10      0.823      0.796     0.814      0.831     0.824
Парные разности, часть B:
       часть   threshold  N  разность  n                          ROC AUC               сбаланс. accuracy                       precision                          recall                   специфичность                        accuracy
B (общие 50) recall 80 % 10 (б) − (а) 10  -0.015 (-0.14…+0.05); >0 в 5/10 -0.049 (-0.17…+0.17); >0 в 2/10 -0.041 (-0.43…+0.39); >0 в 3/10 -0.014 (-0.19…+0.18); >0 в 5/10 -0.083 (-0.53…+0.53); >0 в 3/10 -0.054 (-0.23…+0.23); >0 в 2/10
B (общие 50) recall 80 % 20 (б) − (а) 10  +0.008 (-0.04…+0.04); >0 в 5/10 +0.014 (-0.06…+0.16); >0 в 4/10 +0.087 (-0.06…+0.30); >0 в 6/10 -0.071 (-0.22…+0.05); >0 в 2/10 +0.100 (-0.15…+0.47); >0 в 6/10 +0.028 (-0.07…+0.21); >0 в 4/10
B (общие 50) recall 80 % 20 (в) − (а) 10  +0.021 (-0.02…+0.05); >0 в 8/10 +0.012 (-0.14…+0.16); >0 в 5/10 +0.071 (-0.06…+0.29); >0 в 8/10 -0.076 (-0.31…+0.04); >0 в 1/10 +0.100 (-0.03…+0.43); >0 в 8/10 +0.026 (-0.11…+0.20); >0 в 5/10
B (общие 50) recall 80 % 30 (б) − (а) 10  +0.022 (-0.06…+0.08); >0 в 7/10 -0.049 (-0.10…+0.01); >0 в 1/10 -0.086 (-0.40…+0.05); >0 в 1/10 -0.019 (-0.19…+0.32); >0 в 3/10 -0.079 (-0.46…+0.10); >0 в 3/10 -0.054 (-0.14…+0.00); >0 в 0/10
B (общие 50) recall 80 % 50 (б) − (а) 10 +0.044 (+0.01…+0.07); >0 в 10/10 +0.027 (-0.01…+0.07); >0 в 7/10 +0.062 (-0.05…+0.13); >0 в 8/10 -0.014 (-0.13…+0.16); >0 в 2/10 +0.069 (-0.06…+0.16); >0 в 8/10 +0.034 (-0.02…+0.06); >0 в 8/10
B (общие 50) recall 80 % 50 (в) − (а) 10 +0.050 (+0.02…+0.07); >0 в 10/10 +0.004 (-0.10…+0.06); >0 в 7/10 +0.090 (-0.01…+0.23); >0 в 9/10 -0.081 (-0.28…+0.04); >0 в 1/10 +0.090 (-0.02…+0.23); >0 в 9/10 +0.018 (-0.08…+0.08); >0 в 6/10
B (общие 50) recall 80 % 70 (б) − (а) 10 +0.045 (+0.01…+0.09); >0 в 10/10 +0.003 (-0.06…+0.06); >0 в 6/10 +0.046 (-0.15…+0.21); >0 в 7/10 -0.043 (-0.18…+0.12); >0 в 3/10 +0.048 (-0.13…+0.16); >0 в 7/10 +0.010 (-0.06…+0.07); >0 в 5/10
B (общие 50)        Юден 10 (б) − (а) 10  -0.015 (-0.14…+0.05); >0 в 5/10 -0.082 (-0.25…+0.01); >0 в 1/10 -0.054 (-0.50…+0.24); >0 в 5/10 -0.105 (-0.26…+0.17); >0 в 2/10 -0.059 (-0.51…+0.19); >0 в 5/10 -0.078 (-0.27…+0.03); >0 в 1/10
B (общие 50)        Юден 20 (б) − (а) 10  +0.008 (-0.04…+0.04); >0 в 5/10 -0.018 (-0.08…+0.09); >0 в 2/10 +0.017 (-0.22…+0.25); >0 в 5/10 -0.067 (-0.23…+0.18); >0 в 1/10 +0.031 (-0.23…+0.30); >0 в 5/10 -0.010 (-0.08…+0.13); >0 в 2/10
B (общие 50)        Юден 20 (в) − (а) 10  +0.021 (-0.02…+0.05); >0 в 8/10 -0.022 (-0.15…+0.06); >0 в 3/10 -0.012 (-0.20…+0.13); >0 в 6/10 -0.038 (-0.30…+0.17); >0 в 2/10 -0.007 (-0.22…+0.12); >0 в 6/10 -0.020 (-0.12…+0.07); >0 в 3/10
B (общие 50)        Юден 30 (б) − (а) 10  +0.022 (-0.06…+0.08); >0 в 7/10 -0.046 (-0.13…+0.04); >0 в 3/10 -0.066 (-0.14…+0.05); >0 в 1/10 -0.024 (-0.18…+0.17); >0 в 4/10 -0.069 (-0.22…+0.08); >0 в 1/10 -0.050 (-0.13…+0.02); >0 в 2/10
B (общие 50)        Юден 50 (б) − (а) 10 +0.044 (+0.01…+0.07); >0 в 10/10 +0.022 (-0.05…+0.10); >0 в 6/10 -0.005 (-0.15…+0.13); >0 в 3/10 +0.048 (-0.13…+0.23); >0 в 6/10 -0.003 (-0.15…+0.16); >0 в 3/10 +0.018 (-0.06…+0.08); >0 в 4/10
B (общие 50)        Юден 50 (в) − (а) 10 +0.050 (+0.02…+0.07); >0 в 10/10 +0.021 (-0.10…+0.09); >0 в 7/10 +0.038 (-0.06…+0.12); >0 в 7/10 +0.005 (-0.24…+0.23); >0 в 3/10 +0.038 (-0.06…+0.15); >0 в 6/10 +0.024 (-0.08…+0.08); >0 в 7/10
B (общие 50)        Юден 70 (б) − (а) 10 +0.045 (+0.01…+0.09); >0 в 10/10 +0.007 (-0.11…+0.07); >0 в 7/10 -0.002 (-0.19…+0.22); >0 в 5/10 +0.014 (-0.13…+0.22); >0 в 4/10 -0.000 (-0.27…+0.20); >0 в 5/10 +0.006 (-0.13…+0.09); >0 в 6/10
всего 129.6 мин
```

## 9. Итоги

Модель — REVE v4, ансамбль 3 запусков. Отдельные запуски на всех 121 людях Турции до адаптации: ROC AUC 0,899 / 0,866 / 0,890; NMT eval: 0,863 / 0,891 / 0,875. Путь через кэш совпал с полной моделью (расхождение ≤ 0,0002).

### Часть A: 20 выборок по 20 человек (те же, что в калибровке на месте), проверка на 101

| порог | вариант | ROC AUC | сбаланс. accuracy | precision | recall | специфичность |
|---|---|---|---|---|---|---|
| Юден | (а) только порог | 0,889 (0,87–0,92) | 0,802 (0,71–0,84) | 0,879 | 0,682 | 0,922 |
| Юден | (б) голова | 0,926 (0,85–0,97) | 0,817 (0,69–0,91) | 0,852 | 0,737 | 0,896 |
| Юден | (в) голова + 2 блока | 0,932 (0,87–0,98) | 0,823 (0,72–0,90) | 0,838 | 0,769 | 0,878 |
| recall 80 % | (а) | 0,889 | 0,807 (0,75–0,84) | 0,815 | 0,751 | 0,862 |
| recall 80 % | (б) | 0,926 | 0,829 (0,72–0,91) | 0,819 | 0,796 | 0,861 |
| recall 80 % | (в) | 0,932 | 0,824 (0,72–0,90) | 0,827 | 0,785 | 0,863 |

Парные разности по 20 выборкам (среднее, 2,5–97,5 %, в скольких выборках > 0):
- ROC AUC: (б) − (а) **+0,037** (−0,05…+0,09; 18/20), (в) − (а) **+0,043** (−0,04…+0,09; 19/20).
- Сбаланс. accuracy, Юден: (б) − (а) +0,014 (−0,14…+0,12; 13/20), (в) − (а) +0,021 (−0,10…+0,11; 14/20); recall 80 %: +0,022 (15/20) и +0,017 (12/20).

Дообучение почти всегда улучшает ранжирование (ROC AUC), но пороговые метрики растут лишь на 1–2 п. п. при разбросе ±10–14 п. п.: с 20 людьми порог по оценкам вне выборки (5-кратная проверка на 16 → 4) сам по себе шумный, и выигрыш в AUC до сбалансированной accuracy почти не доходит.

### Часть B: кривая по N, общие 50 проверочных (21 + 29), 10 повторов

Среднее по повторам (рисунок — `results/figures/reve_fewshot_curve.png`, полосы — 2,5–97,5 % повторов):

| N | ROC AUC (а) / (б) / (в) | сбаланс. acc., Юден (а) / (б) / (в) | recall / специфичность при «recall ≥ 80 %», (а) | то же, (б) |
|---|---|---|---|---|
| 0 (порог с NMT) | 0,885 | 0,531 | 0,03 / 1,00 | — |
| 10 | 0,885 / 0,870 / — | 0,836 / 0,754 / — | 0,84 / 0,77 | 0,82 / 0,68 |
| 20 | 0,885 / 0,893 / 0,905 | 0,824 / 0,806 / 0,801 | 0,85 / 0,73 | 0,78 / 0,83 |
| 30 | 0,885 / 0,907 / — | 0,828 / 0,782 / — | 0,81 / 0,84 | 0,79 / 0,76 |
| 50 | 0,885 / 0,929 / 0,935 | 0,812 / 0,834 / 0,834 | 0,83 / 0,80 | 0,82 / 0,87 |
| 70 | 0,885 / 0,930 / — | 0,816 / 0,823 / — | 0,83 / 0,81 | 0,79 / 0,86 |

Парные разности ROC AUC: при N = 10 (б) − (а) −0,015 (5/10), N = 20 +0,008 (5/10), (в) +0,021 (8/10); при **N = 50 (б) +0,044 (+0,01…+0,07; 10/10), (в) +0,050 (+0,02…+0,07; 10/10)**; при N = 70 (б) +0,045 (10/10). Сбаланс. accuracy (Юден): при N = 10 (б) − (а) −0,082 (1/10), N = 30 −0,046 (3/10), N = 50 +0,022 (6/10) и (в) +0,021 (7/10), N = 70 +0,007 (7/10).

- **Без адаптации на другой больнице:** ранжирование не падает (ROC AUC 0,885 на общих 50 против 0,878 на NMT eval), но порог с NMT не переносится: сбаланс. accuracy 0,79 на NMT eval (Юден) → **0,53** на Турции, recall 0,82 → 0,06 (при «recall 80 %»: 0,71 → 0,03). Оценки турецких записей сдвинуты вниз — модель почти всех называет «нормой».
- **С N размеченными:** уже при N = 10 калибровка порога восстанавливает сбаланс. accuracy до 0,84 (Юден; при «recall ≥ 80 %» — recall 0,84, специфичность 0,77), дальше качество по порогу почти не растёт (0,81–0,83 при N = 20–70). Дообучение даёт прирост ROC AUC (+0,04–0,05) только при N ≥ 50, в сбаланс. accuracy — +2 п. п. без устойчивости; при N = 10–30 дообучение головы по порогу хуже калибровки.
- Оговорка: NMT («патология» в целом) и Турция («эпилепсия») — разные задачи, поэтому сдвиг порога — не только доменный эффект: меняются и доля, и тип «положительных» записей.

### Вывод для продукта

**По умолчанию достаточно калибровки порога на 10–20 размеченных записях клиники** — она снимает основную потерю при переносе (сбаланс. accuracy 0,53 → 0,81–0,84) и не ухудшает ранжирование. Дообучение модели на данных клиники имеет смысл только при ≥ 50 размеченных записях: тогда ROC AUC растёт устойчиво (+0,04–0,05 во всех 10 повторах), а пороговые метрики — на ~2 п. п. с большим разбросом. Последние 2 блока трансформера сверх головы добавляют ≤ 0,01 ROC AUC и не меняют пороговые метрики. При 10–30 записях дообучение не рекомендуется: порог по оценкам вне выборки внутри малой выборки шумный, сбаланс. accuracy в среднем ниже, чем у одной калибровки.

Ограничения: один внешний набор (121 человек, 2 класса из одной больницы); настройки дообучения заданы заранее, без подбора; (в) в части B — только N = 20 и 50 (на остальные N не хватило бюджета); 10 повторов — полосы на рисунке по сути от минимума до максимума.

**Затраты:** GPU-сеанс 129,6 мин на T4 (≈2,2 ч) — на ~10 мин больше бюджета 2 ч: ограничение времени в блокноте проверялось только перед необязательными (в) и не прерывало основной прогон (по 42,5 мин на запуск: NMT + кэш ≈ 3,5 мин, (б) 9,7 мин, (в) ≈ 29 мин). Плюс проверка конвейера на процессоре Kaggle (18,7 мин, без GPU).

Файлы: `results/reve_fewshot_summary.csv` (часть A), `reve_fewshot_curve.csv` (часть B), `reve_fewshot_paired_differences.csv`, `reve_fewshot_per_split.csv` (метрики каждого разбиения), `reve_fewshot_scores.csv.gz` (оценки людей по разбиениям, вариантам и запускам), `reve_fewshot_splits.csv`, `reve_fewshot_times.csv`, `results/figures/reve_fewshot_curve.png`. Веса после дообучения не сохранялись.